# Lesson 07 — Conditional logic with `CASE WHEN`

In this lesson:
- Use `CASE WHEN` to pick a value based on a condition
- Categorize rows (e.g. "blockbuster" vs. "hit" vs. "modest")
- Use `CASE WHEN` inside aggregates

Time: 20 minutes.

`CASE WHEN` is SQL's version of an if/else statement. It lets you change what shows up in a column depending on what's in each row — without writing four separate queries.

## Setup

In [ ]:
# Sign in to Google and connect to the Disney dataset.
# Run this once per Colab session.

from google.colab import auth
auth.authenticate_user()

from google.cloud.bigquery import magics
magics.context.project = "sql-with-disney"

print("✓ Ready! Run the query cells below.")

## Basic `CASE WHEN`

The shape:

```sql
CASE
  WHEN [condition] THEN [value if true]
  ELSE [value if no condition matched]
END
```

You can put a `CASE` expression anywhere you'd put a column or value — most often in `SELECT`.

Classify each movie as "classic" (before 2000) or "modern" (2000 or later):

In [ ]:
%%bigquery
SELECT
  title,
  year,
  CASE
    WHEN year < 2000 THEN 'classic'
    ELSE 'modern'
  END AS era
FROM disney_lessons.movies
ORDER BY year
LIMIT 10

You get a new column `era` with the value `'classic'` or `'modern'` based on the year of each row.

The `END` is mandatory — it closes out the `CASE`. The `AS era` gives the new column a name (otherwise it'd be something ugly like `f0_`).

## Multiple `WHEN` branches

You can have as many `WHEN` clauses as you want. They're evaluated **top to bottom**, and the first match wins.

Categorize movies by box office tier:

In [ ]:
%%bigquery
SELECT
  title,
  box_office_millions,
  CASE
    WHEN box_office_millions >= 1000 THEN 'blockbuster'
    WHEN box_office_millions >= 500  THEN 'hit'
    WHEN box_office_millions >= 100  THEN 'modest'
    ELSE 'flop'
  END AS performance
FROM disney_lessons.movies
ORDER BY box_office_millions DESC
LIMIT 15

For each row, SQL checks the conditions in order:
- Box office ≥ 1000? Mark as 'blockbuster'.
- Otherwise, ≥ 500? Mark as 'hit'.
- Otherwise, ≥ 100? Mark as 'modest'.
- Otherwise: 'flop'.

The `ELSE` is technically optional, but if you leave it off and no `WHEN` matches, you get `NULL`. Usually you want an explicit `ELSE`.

## `CASE WHEN` inside aggregates

This is where `CASE WHEN` gets really powerful. You can use it inside `COUNT`, `SUM`, etc. to count or sum *only* rows that match a condition.

How many blockbusters does each studio have?

In [ ]:
%%bigquery
SELECT
  studio,
  COUNT(*) AS total_movies,
  COUNT(CASE WHEN box_office_millions >= 1000 THEN 1 END) AS blockbusters
FROM disney_lessons.movies
GROUP BY studio
ORDER BY blockbusters DESC

How that works: `CASE WHEN ... THEN 1 END` returns `1` when the condition is true, and `NULL` when it isn't (because there's no `ELSE`). `COUNT(...)` then counts only the non-NULL values — i.e., only the blockbusters.

GoogleSQL has a shortcut for this exact pattern called `COUNTIF`:

```sql
COUNTIF(box_office_millions >= 1000) AS blockbusters
```

Both work; `COUNTIF` is shorter when you have it.

## `CASE WHEN` with `SUM`

Same idea — sum a value only for rows matching a condition.

Total animation box office vs. total action box office, in one row:

In [ ]:
%%bigquery
SELECT
  ROUND(SUM(CASE WHEN genre = 'Animation' THEN box_office_millions ELSE 0 END), 0) AS animation_box_office,
  ROUND(SUM(CASE WHEN genre = 'Action' THEN box_office_millions ELSE 0 END), 0) AS action_box_office
FROM disney_lessons.movies

This is a classic "pivot in SQL" pattern — turning a grouping into separate columns of one row.

## Quick reminders

- `CASE WHEN` is just SQL's if/else.
- The first matching `WHEN` wins.
- Don't forget the `END`.
- Read-only as always; `%%bigquery` is still Colab-only plumbing.

## Exercises

### Exercise 1

Show each movie's `title`, `runtime_minutes`, and a new column called `length_category`:
- `'long'` if runtime is 120 or more
- `'medium'` if runtime is between 90 and 119
- `'short'` if runtime is under 90

In [ ]:
# Your query here

### Exercise 2

Show each movie's `title`, `rt_score`, and a `quality` label:
- `'great'` if score is 90 or higher
- `'good'` if score is 70–89
- `'meh'` otherwise

Sort by score, highest first. Limit to 10 rows.

In [ ]:
# Your query here

### Exercise 3

For each studio, count how many of its movies are **blockbusters** (`box_office_millions >= 1000`) vs. **non-blockbusters**. Show `studio`, `blockbusters`, `non_blockbusters`.

Hint: use `COUNTIF` for both.

In [ ]:
# Your query here

---

## Solutions

### Solution 1

In [ ]:
%%bigquery
SELECT
  title,
  runtime_minutes,
  CASE
    WHEN runtime_minutes >= 120 THEN 'long'
    WHEN runtime_minutes >= 90  THEN 'medium'
    ELSE 'short'
  END AS length_category
FROM disney_lessons.movies
LIMIT 15

### Solution 2

In [ ]:
%%bigquery
SELECT
  title,
  rt_score,
  CASE
    WHEN rt_score >= 90 THEN 'great'
    WHEN rt_score >= 70 THEN 'good'
    ELSE 'meh'
  END AS quality
FROM disney_lessons.movies
ORDER BY rt_score DESC
LIMIT 10

### Solution 3

In [ ]:
%%bigquery
SELECT
  studio,
  COUNTIF(box_office_millions >= 1000) AS blockbusters,
  COUNTIF(box_office_millions < 1000)  AS non_blockbusters
FROM disney_lessons.movies
GROUP BY studio
ORDER BY blockbusters DESC

## What you've learned

- ✅ `CASE WHEN ... THEN ... ELSE ... END` — SQL's if/else
- ✅ Multiple `WHEN` branches (first match wins)
- ✅ `CASE WHEN` inside aggregates for conditional counting/summing
- ✅ `COUNTIF` as the shorthand for `COUNT(CASE WHEN ... THEN 1 END)`

## Up next

You've worked with text and numbers. There's one more major data type to learn: **dates**. Lesson 08 introduces the `parks` and `attractions` tables and shows you how to ask questions like *"How long ago did Disneyland open?"* and *"How many attractions were built in the 2010s?"*.

Open `08_dates.ipynb`.